# Silver Geography

This notebook builds the **geography backbone** of the Silver layer from the PSGC 2Q 2026 Bronze table (DS_34) and the reviewed DOT→PSGC crosswalk seed. It also checks that the crosswalk covers every province-level and region-level row in the DOT Bronze tables (DS_05).

| Silver table | One row = | Used by |
| --- | --- | --- |
| `geo_psgc` | one PSGC 2Q 2026 entity (region, province, city, municipality, sub-municipality, barangay), with `region_code`, `parent_province_code` and the reporting unit it rolls up to | PSA and population Silver notebooks (roll-ups), Gold `dim_province` |
| `geo_reporting_unit` | one analysis unit: a province, an HUC that DOT reports separately, the City of Isabela, or NCR | Gold `dim_province`, all province-year Silver tables |
| `map_dot_area_to_psgc` | one printed DOT label (region or level-1 row) per validity period, with its `row_role` and target unit | `02_silver_dot` |

**Unit rules applied here** (PR #43 schema doc, D3 and D8; to be recorded in the decision log):

* A unit is a **province**, an **HUC that DOT reports separately**, the **City of Isabela**, or **NCR**.
* **Lucena** (HUC) is not a unit: DOT counts it inside Quezon, so its PSGC entities roll up to Quezon.
* **NCR** is one unit: its 16 HUCs and Pateros roll up to NCR.
* **Region is always taken from the PSGC code** (DL-003), never from the label printed by a source. Negros Occidental, Negros Oriental and Siquijor are therefore in NIR, and Cotabato City is in BARMM.

**DOT row roles** (from the seed CSV): `province_unit` (loaded as the unit), `add_to_parent` (summed into another unit), `exclude` (not loaded, with `exclude_reason`), `region_total` and `national_total` (reconciliation only).

The notebook runs every data-quality check **before** writing. If a `BLOCK` check fails, results go to `dq_results`, no Silver table is written, and the notebook raises an error.

## Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
import uuid, os, re, hashlib
from datetime import datetime
import pandas as pd

# ── Unity Catalog targets ────────────────────────────────────────
CATALOG = "tuklas_dev"
BRONZE_SCHEMA = "02_bronze"
SILVER_SCHEMA = "03_silver"
DQ_SCHEMA = "06_data_quality"
NOTEBOOK_NAME = "01_silver_geography"

run_id = f"silver_geo_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

# ── Inputs ───────────────────────────────────────────────────────
BRONZE_PSGC = "psgc_ds_34_psgc"                  # DS_34, sheet "PSGC" (02_ingest_psgc)
DOT_TABLE_RE = re.compile(r"^dot_ds_05_(\d{4})$")  # DS_05, one table per year (03_ingest_dot)
PSGC_VERSION = "2Q 2026"
CENSUS_POPULATION_YEAR = 2024                    # PSGC "2024 Population" column

# Crosswalk seed, committed to the repo. The notebook walks up from its own folder to find it.
# Set SEED_PATH_OVERRIDE (e.g. a Volume path) only if the repo files are not available.
SEED_RELATIVE_PATH = "resources/seeds/map_dot_area_to_psgc.csv"
SEED_PATH_OVERRIDE = None

# Bronze column names produced by 02_ingest_psgc (headers converted to snake_case; leading digits get "c_")
PSGC_COLS = {
    "code": "c_10_digit_psgc",
    "name": "name",
    "correspondence_code": "correspondence_code",
    "level": "geographic_level",
    "old_names": "old_names",
    "city_class": "city_class",
    "income_class": "income_classification_dof_do_no_074_2024",
    "urban_rural": "urban_rural_based_on_2020_cph",
    "population": "c_2024_population",
    "status": "status",
}

# ── Unit rules ───────────────────────────────────────────────────
NCR_CODE = "1300000000"
# HUCs that DOT counts inside a province (no separate DOT row): HUC code -> province unit
HUCS_COUNTED_IN_PROVINCE = {"0431200000": "0405600000"}   # City of Lucena -> Quezon
# Cities that are their own unit although they are not HUCs: city code -> PSGC code prefix (5 digits)
INDEPENDENT_CITY_UNITS = {"0990101000": "09901"}          # City of Isabela (PSGC "Not a Province" 0990100000)
# PSGC areas that belong to no province and no unit (recorded, not an error)
UNITLESS_PREFIXES = {"19999": "BARMM Special Geographic Area (municipalities not under a province)"}

EXPECTED_LEVEL_COUNTS = {"Reg": 18, "Prov": 82, "City": 149, "Mun": 1493, "SubMun": 14, "Bgy": 42010}
EXPECTED_UNIT_COUNTS = {"Province": 82, "HUC": 16, "Independent city": 1, "NCR": 1}   # 100 units
# Units DOT does not report (BARMM provinces and Sulu); any other unit without DOT rows is flagged
EXPECTED_UNITS_WITHOUT_DOT = {"0906600000", "1900700000", "1903600000", "1907000000", "1908700000", "1908800000"}
# Units expected to have a year not comparable to 2019 (DL-013: Aklan); anything else is flagged for review
EXPECTED_BREAK_UNITS = {"0600400000"}

VALID_ROW_ROLES = {"province_unit", "add_to_parent", "exclude", "region_total", "national_total"}
VALID_EXCLUDE_REASONS = {"double_count", "no_psgc_unit", "outside_dot_coverage"}

# ── Output tables ────────────────────────────────────────────────
TBL_PSGC = "geo_psgc"
TBL_UNIT = "geo_reporting_unit"
TBL_DOT_MAP = "map_dot_area_to_psgc"

def bronze_table(name): return f"`{CATALOG}`.`{BRONZE_SCHEMA}`.`{name}`"
def silver_table(name): return f"`{CATALOG}`.`{SILVER_SCHEMA}`.`{name}`"
def dq_table(name): return f"`{CATALOG}`.`{DQ_SCHEMA}`.`{name}`"

print(f"Run ID: {run_id}")

## Helpers

* `dq(...)` records one check result in memory; all results are written to `dq_results` at the end of the run, whether or not a check failed.
* `write_silver(...)` overwrites a whole Silver table. Each table is rebuilt from Bronze and the seed on every run, so a rerun gives the same result and never duplicates rows.

In [0]:
dq_results = []

def dq(check_id, table_name, severity, failed_rows, expected, observed, sample_keys=None, notes=""):
    """Record one data-quality check result. failed_rows = 0 means PASS."""
    status = "PASS" if failed_rows == 0 else "FAIL"
    keys = None
    if sample_keys:
        keys = " | ".join(str(k) for k in list(sample_keys)[:20])
    dq_results.append((run_id, NOTEBOOK_NAME, check_id, table_name, severity, status, int(failed_rows),
                       str(expected), str(observed), keys, notes))
    marker = "✓" if status == "PASS" else ("✗" if severity == "BLOCK" else "!")
    print(f"{marker} {check_id:<11} {severity:<6} {status:<4} | {table_name} | expected {expected} | observed {observed}"
          + (f" | e.g. {keys}" if keys else ""))

def sample(df, *cols, n=20):
    return [" / ".join("" if r[c] is None else str(r[c]) for c in cols) for r in df.select(*cols).limit(n).collect()]

def write_silver(df, name):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(silver_table(name)))
    return spark.table(silver_table(name)).count()

spark.sql(f"""
INSERT INTO {silver_table('silver_run_log')}
(run_id, notebook_name, started_at, completed_at, tables_written, rows_written,
 dq_checks_run, dq_block_failures, dq_review_flags, status, notes)
VALUES (:run_id, :nb, current_timestamp(), NULL, 0, 0, 0, 0, 0, 'STARTED', 'Notebook execution started')
""", args={"run_id": run_id, "nb": NOTEBOOK_NAME})
print(f"✓ STARTED record inserted for {run_id}")

## Step 1 — `geo_psgc`: clean the PSGC datafile

From `psgc_ds_34_psgc` (text as printed):

* Codes are trimmed and left-padded (`psgc_code` to 10 digits, `correspondence_code` to 9), so a code saved as a number never loses its leading zero.
* Names are trimmed (the datafile has trailing spaces, e.g. `City of Lucena `).
* `population_2024` is cast to a whole number.
* `region_code` = first 2 digits + `00000000`. `parent_province_code` = first 5 digits + `00000` when that code is a province (NULL for regions, provinces, HUCs and their barangays, and NCR).
* `unit_psgc_code` = the reporting unit the entity rolls up to (rules in the Configuration cell).

The Bronze sheet `Prov Sum` (dated 30 June 2020) is **not** used (DQ-GEO-06).

In [0]:
bronze_psgc = spark.table(bronze_table(BRONZE_PSGC))
missing_cols = [c for c in PSGC_COLS.values() if c not in bronze_psgc.columns]
assert not missing_cols, f"Columns not found in {BRONZE_PSGC}: {missing_cols}. Check the headers recorded in file_manifest notes."

def txt(key):
    """Trimmed text; empty strings become NULL."""
    c = F.trim(F.col(PSGC_COLS[key]))
    return F.when(c == "", None).otherwise(c)

psgc_base = (bronze_psgc
    .select(
        F.lpad(txt("code"), 10, "0").alias("psgc_code"),
        txt("name").alias("psgc_name"),
        F.lpad(txt("correspondence_code"), 9, "0").alias("correspondence_code"),
        txt("level").alias("geographic_level"),
        txt("city_class").alias("city_class"),
        txt("old_names").alias("old_names"),
        txt("income_class").alias("income_class"),
        txt("urban_rural").alias("urban_rural_2020"),
        F.expr(f"try_cast(regexp_replace(trim(`{PSGC_COLS['population']}`), ',', '') AS BIGINT)").alias("population_2024"),
        txt("status").alias("status"),
        F.col("source_row_number").cast("int").alias("source_row_number"),
        "_source_dataset_id", "_source_file", "_source_sha256",
        F.col("_run_id").alias("_bronze_run_id"))
    .withColumn("region_code", F.concat(F.substring("psgc_code", 1, 2), F.lit("00000000")))
    .withColumn("_prov_candidate", F.concat(F.substring("psgc_code", 1, 5), F.lit("00000")))
    .withColumn("_prefix5", F.substring("psgc_code", 1, 5)))

regions = psgc_base.filter("geographic_level = 'Reg'").select(F.col("psgc_code").alias("region_code"), F.col("psgc_name").alias("region_name"))
provinces = psgc_base.filter("geographic_level = 'Prov'").select(F.col("psgc_code").alias("_prov_candidate")).withColumn("_is_prov", F.lit(True))
hucs = [r["psgc_code"] for r in psgc_base.filter("city_class = 'HUC'").select("psgc_code").collect()]
huc_prefix = {c[:5]: c for c in hucs}

# Rolls up to: NCR -> independent cities -> HUCs counted in a province -> province itself -> parent province -> own HUC
unit_expr = F.when(F.col("region_code") == NCR_CODE, F.lit(NCR_CODE))
for city_code, prefix in INDEPENDENT_CITY_UNITS.items():
    unit_expr = unit_expr.when(F.col("_prefix5") == prefix, F.lit(city_code))
for huc_code, prov_code in HUCS_COUNTED_IN_PROVINCE.items():
    unit_expr = unit_expr.when(F.col("_prefix5") == huc_code[:5], F.lit(prov_code))
unit_expr = (unit_expr
    .when(F.col("geographic_level") == "Reg", F.lit(None))
    .when(F.col("geographic_level") == "Prov", F.col("psgc_code"))
    .when(F.col("parent_province_code").isNotNull(), F.col("parent_province_code")))
for prefix, huc_code in huc_prefix.items():
    unit_expr = unit_expr.when(F.col("_prefix5") == prefix, F.lit(huc_code))

geo_psgc = (psgc_base
    .join(regions, "region_code", "left")
    .join(provinces, "_prov_candidate", "left")
    .withColumn("parent_province_code",
                F.when(F.col("geographic_level").isin("Reg", "Prov"), F.lit(None))
                 .when(F.col("_is_prov"), F.col("_prov_candidate")))
    .withColumn("unit_psgc_code", unit_expr.otherwise(F.lit(None)))
    .withColumn("psgc_version", F.lit(PSGC_VERSION))
    .withColumn("_silver_run_id", F.lit(run_id))
    .withColumn("_processed_at", F.current_timestamp())
    .select("psgc_code", "psgc_name", "geographic_level", "city_class", "region_code", "region_name",
            "parent_province_code", "unit_psgc_code", "correspondence_code", "old_names", "income_class",
            "urban_rural_2020", "population_2024", "status", "psgc_version", "source_row_number",
            "_source_dataset_id", "_source_file", "_source_sha256", "_bronze_run_id", "_silver_run_id", "_processed_at"))

print(f"geo_psgc rows: {geo_psgc.count()}")
display(geo_psgc.groupBy("geographic_level").count().orderBy("geographic_level"))

## Step 2 — `geo_reporting_unit`: the 100 analysis units

| `unit_type` | Selected from PSGC | Count |
| --- | --- | --- |
| Province | level `Prov` (includes the 5 BARMM provinces and Sulu, which DOT does not report) | 82 |
| HUC | level `City`, class `HUC`, outside NCR, except Lucena | 16 |
| Independent city | City of Isabela | 1 |
| NCR | the NCR region row | 1 |

`census_population_ref` is the PSGC 2024 census count for the unit's area (the unit's own row, plus Lucena for Quezon). It is **reference only**; annual denominators come from the population Silver table.

`dot_reported` is filled in Step 3 from the crosswalk. `dot_breaks_comparability` is `true` when the unit has at least one year in which DOT counts it differently from 2019 (a seed row with `breaks_comparability = true` rolls up to it); `dot_comparability_note` lists those rows. Currently only Aklan (Boracay from 2021, DL-013); Pampanga stays comparable (DL-014). The **year-level** flag `comparable_to_2019` is built in `02_silver_dot` from the same seed rows (DL-020); this unit-level flag feeds `dim_province.comparability_note`.

In [0]:
unit_rows = (geo_psgc
    .withColumn("unit_type",
        F.when(F.col("psgc_code") == NCR_CODE, F.lit("NCR"))
         .when(F.col("psgc_code").isin(list(INDEPENDENT_CITY_UNITS)), F.lit("Independent city"))
         .when(F.col("geographic_level") == "Prov", F.lit("Province"))
         .when((F.col("city_class") == "HUC") & (F.col("region_code") != NCR_CODE)
               & ~F.col("psgc_code").isin(list(HUCS_COUNTED_IN_PROVINCE)), F.lit("HUC")))
    .filter(F.col("unit_type").isNotNull()))

# Population of HUCs that are counted inside a province is added to that province
extra_pop = (geo_psgc.filter(F.col("psgc_code").isin(list(HUCS_COUNTED_IN_PROVINCE)))
    .groupBy(F.col("unit_psgc_code").alias("psgc_code"))
    .agg(F.sum("population_2024").alias("_extra_pop"),
         F.concat_ws(", ", F.collect_list("psgc_name")).alias("_extra_names")))

geo_reporting_unit = (unit_rows
    .drop("unit_psgc_code")
    .join(extra_pop, "psgc_code", "left")
    .select(
        F.col("psgc_code").alias("unit_psgc_code"),
        F.col("psgc_name").alias("unit_name"),
        "unit_type", "region_code", "region_name",
        (F.col("population_2024") + F.coalesce(F.col("_extra_pop"), F.lit(0))).alias("census_population_ref"),
        F.lit(CENSUS_POPULATION_YEAR).alias("census_population_ref_year"),
        F.when(F.col("_extra_names").isNotNull(), F.concat(F.lit("Includes "), F.col("_extra_names"), F.lit(" (HUC counted inside this province by DOT)")))
         .when(F.col("unit_type") == "NCR", F.lit("NCR is one unit; its cities and Pateros roll up to it"))
         .alias("unit_note"),
        F.lit(PSGC_VERSION).alias("psgc_version"),
        "_source_dataset_id", "_source_file", "_source_sha256", "_bronze_run_id"))

display(geo_reporting_unit.groupBy("unit_type").count())

## Step 3 — `map_dot_area_to_psgc`: load the crosswalk seed

The seed `resources/seeds/map_dot_area_to_psgc.csv` holds one row per printed DOT label (region rows at indent level 0, province-level rows at indent level 1) per validity period. It is reviewed in pull requests like code; this notebook never edits it.

| Seed column | Meaning |
| --- | --- |
| `dot_indent_level`, `dot_parent_label`, `dot_area_label` | The DOT row exactly as printed (parent is the region label for level 1; empty for level 0) |
| `valid_from_year`, `valid_to_year` | Years the rule applies to (empty `valid_to_year` = still in use) |
| `row_role` | `province_unit`, `add_to_parent`, `exclude`, `region_total`, `national_total` |
| `unit_psgc_code` | Unit the travelers are loaded to (for `province_unit` and `add_to_parent`) |
| `area_psgc_code` | PSGC code of the printed area itself (e.g. Malay for Boracay, City of Masbate) |
| `exclude_reason` | `double_count`, `no_psgc_unit`, `outside_dot_coverage` |
| `breaks_comparability` | `true` only on rows treated differently from 2019 (currently Boracay 2021–2022 and Boracay (Malay) 2023–), so `comparable_to_2019` can be derived per unit and year (DL-020) |

Key: (`dot_indent_level`, `dot_parent_label`, `dot_area_label`, `valid_from_year`).

In [0]:
def find_seed_path():
    if SEED_PATH_OVERRIDE:
        return SEED_PATH_OVERRIDE
    here = os.getcwd()
    for _ in range(8):
        candidate = os.path.join(here, SEED_RELATIVE_PATH)
        if os.path.exists(candidate):
            return candidate
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    raise FileNotFoundError(f"{SEED_RELATIVE_PATH} not found above {os.getcwd()}. Run from the repo Git folder or set SEED_PATH_OVERRIDE.")

seed_path = find_seed_path()
with open(seed_path, "rb") as fh:
    seed_bytes = fh.read()
seed_sha256 = hashlib.sha256(seed_bytes).hexdigest()

seed_pd = pd.read_csv(seed_path, dtype=str, keep_default_na=False)
seed_pd = seed_pd.apply(lambda s: s.str.strip()).replace({"": None})
seed_cols = list(seed_pd.columns)
seed_raw = spark.createDataFrame(seed_pd.astype(object).where(seed_pd.notna(), None),
                                 schema=", ".join(f"`{c}` STRING" for c in seed_cols))
print(f"Seed: {seed_path}\n  rows: {len(seed_pd)}  sha256: {seed_sha256[:12]}…")

seed = (seed_raw
    .withColumn("dot_indent_level", F.col("dot_indent_level").cast("int"))
    .withColumn("valid_from_year", F.col("valid_from_year").cast("int"))
    .withColumn("valid_to_year", F.col("valid_to_year").cast("int"))
    .withColumn("breaks_comparability", F.coalesce(F.lower("breaks_comparability") == "true", F.lit(False)))
    .withColumn("reviewed_on", F.to_date("reviewed_on")))

unit_lookup = geo_reporting_unit.select("unit_psgc_code", "unit_name", "unit_type",
                                        F.col("region_code").alias("unit_region_code"),
                                        F.col("region_name").alias("unit_region_name"))
area_lookup = geo_psgc.select(F.col("psgc_code").alias("area_psgc_code"), F.col("psgc_name").alias("area_psgc_name"),
                              F.col("geographic_level").alias("area_geographic_level"))

map_dot = (seed
    .join(unit_lookup, "unit_psgc_code", "left")
    .join(area_lookup, "area_psgc_code", "left")
    .withColumn("_seed_file", F.lit(SEED_RELATIVE_PATH))
    .withColumn("_seed_sha256", F.lit(seed_sha256))
    .withColumn("_silver_run_id", F.lit(run_id))
    .withColumn("_processed_at", F.current_timestamp())
    .select("dot_indent_level", "dot_parent_label", "dot_area_label", "valid_from_year", "valid_to_year",
            "row_role", "unit_psgc_code", "unit_name", "unit_type", "unit_region_code", "unit_region_name",
            "area_psgc_code", "area_psgc_name", "area_geographic_level", "exclude_reason", "breaks_comparability",
            "match_method", "match_status", "notes", "reviewed_by", "reviewed_on",
            "_seed_file", "_seed_sha256", "_silver_run_id", "_processed_at"))

display(map_dot.groupBy("row_role").count().orderBy("row_role"))

# Unit-level comparability: a seed row flagged breaks_comparability marks the unit it belongs to.
# Rows with no unit_psgc_code (e.g. Boracay 2021–24, excluded as double count) resolve through the
# PSGC unit of the printed area (Malay -> Aklan).
area_unit = geo_psgc.select(F.col("psgc_code").alias("area_psgc_code"), F.col("unit_psgc_code").alias("_area_unit"))
breaks_rows = (map_dot.filter("breaks_comparability")
    .join(area_unit, "area_psgc_code", "left")
    .withColumn("_unit", F.coalesce("unit_psgc_code", "_area_unit"))
    .withColumn("_desc", F.concat("dot_area_label", F.lit(" "), F.col("valid_from_year").cast("string"), F.lit("–"),
                                  F.coalesce(F.col("valid_to_year").cast("string"), F.lit("")), F.lit(" ("), "row_role", F.lit(")"))))
unit_breaks = (breaks_rows.filter(F.col("_unit").isNotNull())
    .groupBy(F.col("_unit").alias("unit_psgc_code"))
    .agg(F.concat(F.lit("DOT reporting changed across years: "),
                  F.concat_ws("; ", F.array_sort(F.collect_set("_desc")))).alias("dot_comparability_note")))
display(unit_breaks)

## Step 4 — Match the DOT Bronze rows to the crosswalk

Every DOT Bronze table `dot_ds_05_<year>` is read; only indent levels 0 (regions, `GRAND TOTAL`) and 1 (provinces, HUCs, cities, zones) are matched. Level-2 rows are sub-areas already counted in their parent and are never loaded (DQ-DOT-04).

A row matches a seed row when indent level, parent label, area label are equal and the year is within `valid_from_year`…`valid_to_year`.

In [0]:
dot_tables = sorted(r["tableName"] for r in spark.sql(f"SHOW TABLES IN `{CATALOG}`.`{BRONZE_SCHEMA}`").collect()
                    if DOT_TABLE_RE.match(r["tableName"]))
print(f"DOT Bronze tables: {dot_tables}")

dot_rows = None
for t in dot_tables:
    df = (spark.table(bronze_table(t))
          .select(F.col("source_year").cast("int").alias("year"),
                  F.col("indent_level").cast("int").alias("indent_level"),
                  F.trim("parent_label").alias("parent_label"),
                  F.trim("area_label").alias("area_label"),
                  F.col("total_travelers"),
                  F.lit(t).alias("bronze_table"))
          .filter(F.col("indent_level").isin(0, 1)))
    dot_rows = df if dot_rows is None else dot_rows.unionByName(df)

if dot_rows is not None:
    dot_rows = (dot_rows
        .withColumn("parent_label", F.when(F.col("indent_level") == 0, F.lit(None)).otherwise(F.col("parent_label")))
        .withColumn("total_num", F.expr("CASE WHEN trim(total_travelers) = '-' THEN 0 "
                                         "ELSE try_cast(regexp_replace(total_travelers, ',', '') AS BIGINT) END"))
        .withColumn("row_key", F.concat_ws("|", "bronze_table", "indent_level", F.coalesce("parent_label", F.lit("")), "area_label")))

    s = map_dot.alias("s")
    d = dot_rows.alias("d")
    dot_matched = (d.join(s,
            (F.col("d.indent_level") == F.col("s.dot_indent_level"))
            & F.col("d.parent_label").eqNullSafe(F.col("s.dot_parent_label"))
            & (F.col("d.area_label") == F.col("s.dot_area_label"))
            & (F.col("d.year") >= F.col("s.valid_from_year"))
            & (F.col("d.year") <= F.coalesce(F.col("s.valid_to_year"), F.lit(9999))),
            "left")
        .select("d.*", "s.dot_indent_level", "s.dot_parent_label", "s.dot_area_label", "s.valid_from_year",
                "s.row_role", "s.unit_psgc_code", "s.exclude_reason"))
    dot_years = sorted(r["year"] for r in dot_rows.select("year").distinct().collect())
    print(f"DOT years: {dot_years} | level 0/1 rows: {dot_rows.count()}")
else:
    dot_matched, dot_years = None, []

## Step 5 — Data-quality checks

IDs follow `docs/data-quality-rules.md` (Issue #5). New crosswalk checks use the `DQ-XW` prefix.

| ID | Check | Severity |
| --- | --- | --- |
| DQ-GEO-01 | PSGC codes are 10 digits and unique | BLOCK |
| DQ-GEO-02 | Entity counts match the 2Q 2026 release | REVIEW |
| DQ-GEO-03 | One PSGC source file (one version) | BLOCK |
| DQ-GEO-04 | Every city, municipality, sub-municipality and barangay rolls up to a unit (except listed unitless areas) | BLOCK |
| DQ-GEO-05 | Every `region_code` is a PSGC region | BLOCK |
| DQ-GEO-07 | 100 reporting units, unique, with the expected type counts | BLOCK |
| DQ-GEO-08 | Every unit has a census population reference | REVIEW |
| DQ-COM-02 | No NULL code or name | BLOCK |
| DQ-COM-09 | No Unicode replacement characters (`�`) in names | BLOCK |
| DQ-XW-01 | Seed key unique; validity periods for the same label do not overlap | BLOCK |
| DQ-XW-02 | `row_role` and `exclude_reason` valid; required fields present for each role | BLOCK |
| DQ-XW-03 | Seed `unit_psgc_code` is a reporting unit and `area_psgc_code` is a PSGC code | BLOCK |
| DQ-XW-04 | Seed rows that match no DOT Bronze row | LOG |
| DQ-XW-05 | Every `breaks_comparability` seed row resolves to a reporting unit (listed units are flagged `dot_breaks_comparability`) | BLOCK |
| DQ-XW-06 | The units flagged `dot_breaks_comparability` are exactly the expected ones (currently Aklan) | REVIEW |
| DQ-DOT-03 | Every DOT level-0/1 Bronze row matches exactly one seed row | BLOCK |
| DQ-DOT-05 | At most one `province_unit` row per unit per year | BLOCK |
| DQ-DOT-09 | Units without DOT rows are only the BARMM provinces and Sulu | REVIEW |
| DQ-DOT-15 | Per year, loaded rows + non-double-count exclusions = DOT `GRAND TOTAL` (`-` counted as 0) | BLOCK |

In [0]:
# ── Geography ──
bad_code = geo_psgc.filter(~F.col("psgc_code").rlike(r"^\d{10}$") | F.col("psgc_code").isNull())
dup_code = geo_psgc.groupBy("psgc_code").count().filter("count > 1")
dq("DQ-GEO-01", TBL_PSGC, "BLOCK", bad_code.count() + dup_code.count(), "0 invalid or duplicate codes",
   f"{bad_code.count()} invalid, {dup_code.count()} duplicated",
   sample(bad_code, "source_row_number", "psgc_code") + sample(dup_code, "psgc_code"))

level_counts = {r["geographic_level"]: r["count"] for r in geo_psgc.groupBy("geographic_level").count().collect()}
level_diff = {k: (v, level_counts.get(k, 0)) for k, v in EXPECTED_LEVEL_COUNTS.items() if level_counts.get(k, 0) != v}
dq("DQ-GEO-02", TBL_PSGC, "REVIEW", len(level_diff), EXPECTED_LEVEL_COUNTS,
   {k: level_counts.get(k, 0) for k in EXPECTED_LEVEL_COUNTS} | {"(no level)": level_counts.get(None, 0)},
   [f"{k}: expected {e}, got {o}" for k, (e, o) in level_diff.items()])

n_files = bronze_psgc.select("_source_file", "_source_sha256").distinct().count()
dq("DQ-GEO-03", TBL_PSGC, "BLOCK", 0 if n_files == 1 else n_files, "1 source file", f"{n_files} file(s)")

unitless_cond = F.lit(False)
for prefix in UNITLESS_PREFIXES:
    unitless_cond = unitless_cond | (F.substring("psgc_code", 1, 5) == prefix)
no_unit = geo_psgc.filter(F.col("geographic_level").isin("City", "Mun", "SubMun", "Bgy")
                          & F.col("unit_psgc_code").isNull() & ~unitless_cond)
unitless_known = geo_psgc.filter(F.col("unit_psgc_code").isNull() & unitless_cond).count()
dq("DQ-GEO-04", TBL_PSGC, "BLOCK", no_unit.count(), "0 entities without a unit", f"{no_unit.count()} without a unit",
   sample(no_unit, "psgc_code", "psgc_name"),
   notes=f"{unitless_known} rows in known unitless areas: {'; '.join(UNITLESS_PREFIXES.values())}")

bad_region = geo_psgc.filter(F.col("region_name").isNull())
dq("DQ-GEO-05", TBL_PSGC, "BLOCK", bad_region.count(), "every region_code is a PSGC region",
   f"{bad_region.count()} rows with unknown region", sample(bad_region, "psgc_code", "region_code"))

unit_counts = {r["unit_type"]: r["count"] for r in geo_reporting_unit.groupBy("unit_type").count().collect()}
n_units, n_unit_keys = geo_reporting_unit.count(), geo_reporting_unit.select("unit_psgc_code").distinct().count()
unit_fail = (n_units != n_unit_keys) or unit_counts != EXPECTED_UNIT_COUNTS
dq("DQ-GEO-07", TBL_UNIT, "BLOCK", 1 if unit_fail else 0, EXPECTED_UNIT_COUNTS, unit_counts | {"distinct keys": n_unit_keys})

no_pop = geo_reporting_unit.filter(F.col("census_population_ref").isNull())
dq("DQ-GEO-08", TBL_UNIT, "REVIEW", no_pop.count(), "0 units without population", f"{no_pop.count()}",
   sample(no_pop, "unit_psgc_code", "unit_name"))

null_keys = geo_psgc.filter(F.col("psgc_code").isNull() | F.col("psgc_name").isNull())
dq("DQ-COM-02", TBL_PSGC, "BLOCK", null_keys.count(), "0 NULL codes or names", f"{null_keys.count()}",
   sample(null_keys, "source_row_number", "psgc_code"))

bad_chars = geo_psgc.filter(F.col("psgc_name").contains("�") | F.col("old_names").contains("�"))
dq("DQ-COM-09", TBL_PSGC, "BLOCK", bad_chars.count(), "0 names with �", f"{bad_chars.count()}",
   sample(bad_chars, "psgc_code", "psgc_name"))

# ── Crosswalk seed ──
key_cols = ["dot_indent_level", "dot_parent_label", "dot_area_label", "valid_from_year"]
dup_seed = map_dot.groupBy(*key_cols).count().filter("count > 1")
a, b = map_dot.alias("a"), map_dot.alias("b")
overlap = (a.join(b,
        (F.col("a.dot_indent_level") == F.col("b.dot_indent_level"))
        & F.col("a.dot_parent_label").eqNullSafe(F.col("b.dot_parent_label"))
        & (F.col("a.dot_area_label") == F.col("b.dot_area_label"))
        & (F.col("a.valid_from_year") < F.col("b.valid_from_year"))
        & (F.coalesce(F.col("a.valid_to_year"), F.lit(9999)) >= F.col("b.valid_from_year")))
    .select(F.col("a.dot_area_label").alias("dot_area_label"), F.col("a.valid_from_year").alias("from_a"),
            F.col("b.valid_from_year").alias("from_b")))
n_bad_seed = dup_seed.count() + overlap.count()
bad_years = map_dot.filter(F.col("valid_from_year").isNull()
                           | (F.coalesce(F.col("valid_to_year"), F.lit(9999)) < F.col("valid_from_year")))
dq("DQ-XW-01", TBL_DOT_MAP, "BLOCK", n_bad_seed + bad_years.count(), "unique keys, no overlapping periods",
   f"{dup_seed.count()} duplicate keys, {overlap.count()} overlaps, {bad_years.count()} invalid year ranges",
   sample(dup_seed, "dot_area_label", "valid_from_year") + sample(overlap, "dot_area_label", "from_a", "from_b")
   + sample(bad_years, "dot_area_label", "valid_from_year"))

role_bad = map_dot.filter(
    ~F.col("row_role").isin(list(VALID_ROW_ROLES)) | F.col("row_role").isNull()
    | (F.col("row_role").isin("province_unit", "add_to_parent") & F.col("unit_psgc_code").isNull())
    | ((F.col("row_role") == "exclude") & (~F.col("exclude_reason").isin(list(VALID_EXCLUDE_REASONS)) | F.col("exclude_reason").isNull()))
    | ((F.col("row_role") != "exclude") & F.col("exclude_reason").isNotNull())
    | (F.col("row_role").isin("region_total", "national_total") & (F.col("dot_indent_level") != 0))
    | (F.col("row_role").isin("province_unit", "add_to_parent", "exclude") & (F.col("dot_indent_level") != 1)))
dq("DQ-XW-02", TBL_DOT_MAP, "BLOCK", role_bad.count(), "valid roles with required fields", f"{role_bad.count()} invalid rows",
   sample(role_bad, "dot_area_label", "row_role", "unit_psgc_code", "exclude_reason"))

bad_unit = map_dot.filter(F.col("unit_psgc_code").isNotNull() & F.col("unit_name").isNull())
bad_area = map_dot.filter(F.col("area_psgc_code").isNotNull() & F.col("area_psgc_name").isNull())
dq("DQ-XW-03", TBL_DOT_MAP, "BLOCK", bad_unit.count() + bad_area.count(), "all codes found",
   f"{bad_unit.count()} unknown unit codes, {bad_area.count()} unknown area codes",
   sample(bad_unit, "dot_area_label", "unit_psgc_code") + sample(bad_area, "dot_area_label", "area_psgc_code"))

# ── DOT Bronze vs crosswalk ──
if dot_matched is None:
    dq("DQ-DOT-03", "dot_ds_05_*", "BLOCK", 1, "DOT Bronze tables present", "no dot_ds_05_<year> tables found")
else:
    match_counts = dot_matched.groupBy("row_key", "year", "parent_label", "area_label").agg(F.count("row_role").alias("n"))
    unmatched = match_counts.filter("n = 0")
    multi = match_counts.filter("n > 1")
    dq("DQ-DOT-03", "dot_ds_05_*", "BLOCK", unmatched.count() + multi.count(), "every level-0/1 row matches exactly 1 seed row",
       f"{unmatched.count()} unmatched, {multi.count()} matched more than once",
       sample(unmatched, "year", "parent_label", "area_label") + sample(multi, "year", "area_label"),
       notes="A new or renamed DOT label must be added to resources/seeds/map_dot_area_to_psgc.csv in a reviewed PR.")

    dup_unit = (dot_matched.filter("row_role = 'province_unit'")
                .groupBy("year", "unit_psgc_code").count().filter("count > 1"))
    dq("DQ-DOT-05", TBL_DOT_MAP, "BLOCK", dup_unit.count(), "≤ 1 province_unit row per unit-year", f"{dup_unit.count()}",
       sample(dup_unit, "year", "unit_psgc_code"))

    loaded_units = dot_matched.filter(F.col("row_role").isin("province_unit", "add_to_parent")).select("year", "unit_psgc_code").distinct()
    year_units = geo_reporting_unit.select("unit_psgc_code").crossJoin(spark.createDataFrame([(y,) for y in dot_years], "year INT"))
    missing = year_units.join(loaded_units, ["year", "unit_psgc_code"], "left_anti")
    unexpected = missing.filter(~F.col("unit_psgc_code").isin(list(EXPECTED_UNITS_WITHOUT_DOT)))
    dq("DQ-DOT-09", TBL_UNIT, "REVIEW", unexpected.count(),
       f"only BARMM provinces and Sulu without DOT rows ({len(EXPECTED_UNITS_WITHOUT_DOT)} per year)",
       f"{missing.count()} unit-years without DOT rows, {unexpected.count()} unexpected",
       sample(unexpected.orderBy("year", "unit_psgc_code"), "year", "unit_psgc_code"))

    recon = (dot_matched.groupBy("year").agg(
                F.sum(F.when(F.col("row_role") == "national_total", F.col("total_num"))).alias("grand_total"),
                F.sum(F.when(F.col("row_role").isin("province_unit", "add_to_parent"), F.col("total_num"))).alias("loaded"),
                F.sum(F.when((F.col("row_role") == "exclude") & (F.col("exclude_reason") != "double_count"), F.col("total_num"))).alias("excluded_not_double"),
                F.sum(F.when(F.col("total_num").isNull(), 1).otherwise(0)).alias("uncastable"))
             .withColumn("difference", F.col("loaded") + F.col("excluded_not_double") - F.col("grand_total"))
             .orderBy("year"))
    display(recon)
    recon_bad = recon.filter((F.col("difference") != 0) | F.col("difference").isNull() | (F.col("uncastable") > 0))
    dq("DQ-DOT-15", "dot_ds_05_*", "BLOCK", recon_bad.count(), "difference = 0 for every year",
       "; ".join(f"{r['year']}: {r['difference']}" for r in recon.collect()),
       sample(recon_bad, "year", "difference", "uncastable"),
       notes="Validates the seed row roles. '-' is counted as 0 here only for reconciliation.")

    used = dot_matched.filter(F.col("row_role").isNotNull()).select(*key_cols).distinct().alias("u")
    never = map_dot.alias("m").join(used,
        (F.col("m.dot_indent_level") == F.col("u.dot_indent_level"))
        & F.col("m.dot_parent_label").eqNullSafe(F.col("u.dot_parent_label"))
        & (F.col("m.dot_area_label") == F.col("u.dot_area_label"))
        & (F.col("m.valid_from_year") == F.col("u.valid_from_year")), "left_anti")
    dq("DQ-XW-04", TBL_DOT_MAP, "LOG", never.count(), "0 unused seed rows", f"{never.count()} unused",
       sample(never, "dot_area_label", "valid_from_year"))

no_unit_break = breaks_rows.filter(F.col("_unit").isNull())
flagged_units = [r["unit_psgc_code"] for r in unit_breaks.orderBy("unit_psgc_code").collect()]
dq("DQ-XW-05", TBL_DOT_MAP, "BLOCK", no_unit_break.count(), "every breaks_comparability row resolves to a unit",
   f"{no_unit_break.count()} unresolved; units flagged: {flagged_units}",
   sample(no_unit_break, "dot_area_label", "valid_from_year"))

unexpected_breaks = set(flagged_units) ^ EXPECTED_BREAK_UNITS
dq("DQ-XW-06", TBL_UNIT, "REVIEW", len(unexpected_breaks), f"flagged units = {sorted(EXPECTED_BREAK_UNITS)}",
   f"flagged: {flagged_units}", sorted(unexpected_breaks),
   notes="A difference means a seed row's breaks_comparability changed; confirm it with a decision-log entry.")

dq_df = spark.createDataFrame(dq_results,
    "run_id STRING, notebook_name STRING, check_id STRING, table_name STRING, severity STRING, status STRING, "
    "failed_rows BIGINT, expected STRING, observed STRING, sample_keys STRING, notes STRING"
    ).withColumn("checked_at", F.current_timestamp())
display(dq_df.select("check_id", "severity", "status", "failed_rows", "expected", "observed", "sample_keys"))

## Step 6 — Write Silver tables and update the run log

* All check results are appended to `dq_results`.
* If any `BLOCK` check failed: nothing is written, the run is logged as `FAILED`, and the notebook raises an error (the previous Silver tables stay as they were).
* Otherwise the three tables are overwritten, and `dot_reported` is set on `geo_reporting_unit`.

In [0]:
dq_df.select("run_id", "notebook_name", "check_id", "table_name", "severity", "status", "failed_rows",
             "expected", "observed", "sample_keys", "checked_at", "notes") \
     .write.format("delta").mode("append").saveAsTable(dq_table("dq_results"))

block_failures = [r for r in dq_results if r[4] == "BLOCK" and r[5] == "FAIL"]
review_flags = [r for r in dq_results if r[4] == "REVIEW" and r[5] == "FAIL"]

def update_run_log(status, tables_written, rows_written, notes):
    spark.sql(f"""
    UPDATE {silver_table('silver_run_log')}
    SET status = :status, completed_at = current_timestamp(), tables_written = :tw, rows_written = :rw,
        dq_checks_run = :dc, dq_block_failures = :db, dq_review_flags = :dr, notes = :notes
    WHERE run_id = :run_id
    """, args={"status": status, "tw": tables_written, "rw": rows_written, "dc": len(dq_results),
               "db": len(block_failures), "dr": len(review_flags), "notes": notes, "run_id": run_id})

if block_failures:
    msg = "BLOCK checks failed: " + ", ".join(r[2] for r in block_failures) + ". No Silver tables written."
    update_run_log("FAILED", 0, 0, msg)
    raise AssertionError(msg + f" See {dq_table('dq_results')} for run_id = '{run_id}'.")

if dot_matched is not None:
    reported = (dot_matched.filter(F.col("row_role").isin("province_unit", "add_to_parent"))
                .groupBy("unit_psgc_code").agg(F.min("year").alias("dot_first_year"), F.max("year").alias("dot_last_year")))
else:
    reported = spark.createDataFrame([], "unit_psgc_code STRING, dot_first_year INT, dot_last_year INT")

geo_reporting_unit_out = (geo_reporting_unit
    .join(reported, "unit_psgc_code", "left")
    .withColumn("dot_reported", F.col("dot_first_year").isNotNull())
    .join(unit_breaks, "unit_psgc_code", "left")
    .withColumn("dot_breaks_comparability", F.col("dot_comparability_note").isNotNull())
    .withColumn("_silver_run_id", F.lit(run_id))
    .withColumn("_processed_at", F.current_timestamp())
    .select("unit_psgc_code", "unit_name", "unit_type", "region_code", "region_name", "census_population_ref",
            "census_population_ref_year", "dot_reported", "dot_first_year", "dot_last_year",
            "dot_breaks_comparability", "dot_comparability_note", "unit_note", "psgc_version",
            "_source_dataset_id", "_source_file", "_source_sha256", "_bronze_run_id", "_silver_run_id", "_processed_at"))

written = {
    TBL_PSGC: write_silver(geo_psgc, TBL_PSGC),
    TBL_UNIT: write_silver(geo_reporting_unit_out, TBL_UNIT),
    TBL_DOT_MAP: write_silver(map_dot, TBL_DOT_MAP),
}
for tbl, n in written.items():
    print(f"✓ {silver_table(tbl)}: {n} rows")

final_status = "SUCCESS_WITH_REVIEW" if review_flags else "SUCCESS"
final_notes = (f"Silver geography: {', '.join(f'{k}={v}' for k, v in written.items())}; "
               f"{len(dq_results)} checks, {len(review_flags)} for review"
               + (f" ({', '.join(r[2] for r in review_flags)})" if review_flags else "") + ".")
update_run_log(final_status, len(written), sum(written.values()), final_notes)

print(f"Run {run_id}: {final_status}\n{final_notes}")
display(spark.table(silver_table("silver_run_log")).filter(F.col("run_id") == run_id))

## Preview

In [0]:
display(spark.table(silver_table(TBL_UNIT)).orderBy("region_code", "unit_type", "unit_name"))
display(spark.table(silver_table(TBL_DOT_MAP)).filter("row_role NOT IN ('province_unit')").orderBy("row_role", "dot_area_label"))